# Save and Catch — PySpark + GCS + BigQuery

Ordem de execução:
1. Testar credenciais e conexões (acessar/consultar).
2. Salvar dados (local, GCS e BigQuery).
3. Excluir artefatos de teste/execução.

In [ ]:
# Opcional: instalar dependências (descomente se precisar)
%pip install pyspark google-cloud-storage google-cloud-bigquery google-auth pandas pyarrow

In [1]:
import uuid
from datetime import datetime
from pathlib import Path

import google.auth
from google.auth.transport.requests import Request
from google.cloud import storage
from google.cloud import bigquery
from pyspark.sql import SparkSession

# ===== AJUSTE AQUI =====
PROJECT_ID = "global-grammar-432121-d7"
BUCKET_NAME = "my_bucket_class_by_vs_code"
BQ_DATASET = "cars_data"
BQ_TABLE = "housing_data_save_and_catch_test"
BQ_LOCATION = "US"  # ex: US, southamerica-east1

LOCAL_CSV_PATH = Path(r"C:\Users\marco\Documents\code_classes\GCP-Data-Engineering-Foundations\data\Housing.csv")
LOCAL_WORK_DIR = Path(r"C:\Users\marco\Documents\code_classes\GCP-Data-Engineering-Foundations\save and catch")
LOCAL_SPARK_OUTPUT = LOCAL_WORK_DIR / "housing_parquet_spark"

RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
GCS_PREFIX = f"save-and-catch/{RUN_ID}"
GCS_CSV_OBJECT = f"{GCS_PREFIX}/Housing.csv"
GCS_PARQUET_PREFIX = f"{GCS_PREFIX}/housing_parquet_spark"

FULL_TABLE_ID = f"{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}"
BQ_TEMP_TABLE_ID = f"{PROJECT_ID}.{BQ_DATASET}.permission_test_{uuid.uuid4().hex[:8]}"

assert LOCAL_CSV_PATH.exists(), f"Arquivo não encontrado: {LOCAL_CSV_PATH}"
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Configuração carregada.")
print(f"Projeto: {PROJECT_ID}")
print(f"CSV local: {LOCAL_CSV_PATH}")
print(f"Diretório trabalho: {LOCAL_WORK_DIR}")
print(f"Destino BQ: {FULL_TABLE_ID}")

Configuração carregada.
Projeto: global-grammar-432121-d7
CSV local: C:\Users\marco\Documents\code_classes\GCP-Data-Engineering-Foundations\data\Housing.csv
Diretório trabalho: C:\Users\marco\Documents\code_classes\GCP-Data-Engineering-Foundations\save and catch
Destino BQ: global-grammar-432121-d7.cars_data.housing_data_save_and_catch_test


C:\Users\marco\AppData\Local\Temp\ipykernel_21240\581959450.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


In [2]:
# 1) TESTE DE CREDENCIAIS E CONEXÕES (acessar + consultar)
credentials, detected_project = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
credentials.refresh(Request())

# Resolve PROJECT_ID: usa o valor informado; se placeholder, usa o detectado pelas credenciais
if PROJECT_ID in ("", "SEU_PROJECT_ID", None):
    PROJECT_ID = detected_project

if not PROJECT_ID:
    raise ValueError(
        "PROJECT_ID não definido. Preencha PROJECT_ID na célula 3 ou configure ADC com projeto padrão."
    )

# Recalcula variáveis dependentes para as próximas etapas
FULL_TABLE_ID = f"{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}"
BQ_TEMP_TABLE_ID = f"{PROJECT_ID}.{BQ_DATASET}.permission_test_{uuid.uuid4().hex[:8]}"

print("ADC OK. Projeto detectado:", detected_project)
print("Projeto efetivo em uso:", PROJECT_ID)

storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)
bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

# GCS: acessar bucket + consultar metadata
bucket = storage_client.bucket(BUCKET_NAME)
bucket.reload()
print("GCS OK. Bucket acessível:", bucket.name)

# BigQuery: consultar
result = bq_client.query("SELECT 1 AS ok").result()
print("BigQuery query OK:", [dict(r.items()) for r in result])

ADC OK. Projeto detectado: global-grammar-432121-d7
Projeto efetivo em uso: global-grammar-432121-d7
GCS OK. Bucket acessível: my_bucket_class_by_vs_code
BigQuery query OK: [{'ok': 1}]


In [3]:
# 1.1) TESTE DE PERMISSÕES (salvar, alterar, excluir) no GCS e BQ

# GCS: salvar + alterar metadata + excluir
gcs_perm_object = f"{GCS_PREFIX}/permission_test.txt"
blob = bucket.blob(gcs_perm_object)
blob.upload_from_string("permission test", content_type="text/plain")
blob.metadata = {"updated_at": datetime.utcnow().isoformat()}
blob.patch()
blob.delete()
print("Permissões GCS OK (save/update/delete).")

# BQ: criar tabela temporária + insert + update + delete + drop
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}")
dataset_ref.location = BQ_LOCATION
bq_client.create_dataset(dataset_ref, exists_ok=True)

bq_client.query(f"""
CREATE TABLE `{BQ_TEMP_TABLE_ID}` (
  id STRING,
  status STRING,
  created_at TIMESTAMP
)
""").result()

bq_client.query(f"INSERT INTO `{BQ_TEMP_TABLE_ID}` (id, status, created_at) VALUES ('1', 'created', CURRENT_TIMESTAMP())").result()
bq_client.query(f"UPDATE `{BQ_TEMP_TABLE_ID}` SET status='updated' WHERE id='1'").result()
bq_client.query(f"DELETE FROM `{BQ_TEMP_TABLE_ID}` WHERE id='1'").result()
bq_client.delete_table(BQ_TEMP_TABLE_ID, not_found_ok=True)
print("Permissões BigQuery OK (save/update/delete).")

C:\Users\marco\AppData\Local\Temp\ipykernel_21240\3792258289.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  blob.metadata = {"updated_at": datetime.utcnow().isoformat()}


Permissões GCS OK (save/update/delete).
Permissões BigQuery OK (save/update/delete).


In [4]:
# 2) SALVAR — primeiro com PySpark local e depois abrir novamente
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path


def ensure_java_for_pyspark():
    java_cmd = shutil.which("java")
    if java_cmd:
        print("Java encontrado no PATH:", java_cmd)
        return

    candidates = []
    for base in [Path(r"C:\Program Files\Java"), Path(r"C:\Program Files\Eclipse Adoptium")]:
        if base.exists():
            candidates.extend(base.rglob("bin/java.exe"))

    if candidates:
        java_exe = sorted(candidates)[-1]
        java_home = str(java_exe.parent.parent)
        os.environ["JAVA_HOME"] = java_home
        os.environ["PATH"] = f"{java_exe.parent};" + os.environ.get("PATH", "")
        print("JAVA_HOME configurado automaticamente:", java_home)
        return

    raise RuntimeError(
        "Java não encontrado. Instale JDK 17 e reinicie o VS Code.\n"
        "Opção rápida (PowerShell admin): winget install EclipseAdoptium.Temurin.17.JDK"
    )


def ensure_hadoop_windows_binaries(local_work_dir: Path):
    hadoop_home = local_work_dir / "hadoop"
    bin_dir = hadoop_home / "bin"
    bin_dir.mkdir(parents=True, exist_ok=True)

    required = {
        "winutils.exe": "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.3.5/bin/winutils.exe",
        "hadoop.dll": "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.3.5/bin/hadoop.dll",
    }

    for filename, url in required.items():
        dst = bin_dir / filename
        if not dst.exists():
            urllib.request.urlretrieve(url, str(dst))
            print(f"{filename} baixado em: {dst}")

    os.environ["HADOOP_HOME"] = str(hadoop_home)
    os.environ["hadoop.home.dir"] = str(hadoop_home)
    os.environ["PATH"] = f"{bin_dir};" + os.environ.get("PATH", "")
    print("HADOOP_HOME configurado em:", hadoop_home)


ensure_java_for_pyspark()
ensure_hadoop_windows_binaries(LOCAL_WORK_DIR)
print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode("utf-8", errors="ignore"))

# Garante execução local do Spark (evita herdar MASTER inválido do ambiente)
os.environ.pop("MASTER", None)
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
    .appName("save-and-catch-housing")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

spark_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(LOCAL_CSV_PATH))
)

# limpeza local da pasta de output
if LOCAL_SPARK_OUTPUT.exists():
    for p in sorted(LOCAL_SPARK_OUTPUT.rglob("*"), reverse=True):
        if p.is_file():
            p.unlink()
    for p in sorted(LOCAL_SPARK_OUTPUT.rglob("*"), reverse=True):
        if p.is_dir():
            p.rmdir()
    if LOCAL_SPARK_OUTPUT.exists():
        LOCAL_SPARK_OUTPUT.rmdir()

spark_df.write.mode("overwrite").parquet(str(LOCAL_SPARK_OUTPUT))
print("Salvou parquet local em:", LOCAL_SPARK_OUTPUT)

spark_df_reopened = spark.read.parquet(str(LOCAL_SPARK_OUTPUT))
print("Reabertura local OK. Linhas:", spark_df_reopened.count())
spark_df_reopened.show(5, truncate=False)

JAVA_HOME configurado automaticamente: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
HADOOP_HOME configurado em: C:\Users\marco\Documents\code_classes\GCP-Data-Engineering-Foundations\save and catch\hadoop
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)

Salvou parquet local em: C:\Users\marco\Documents\code_classes\GCP-Data-Engineering-Foundations\save and catch\housing_parquet_spark
Reabertura local OK. Linhas: 545
+--------+----+--------+---------+-------+--------+---------+--------+---------------+---------------+-------+--------+----------------+
|price   |area|bedrooms|bathrooms|stories|mainroad|guestroom|basement|hotwaterheating|airconditioning|parking|prefarea|furnishingstatus|
+--------+----+--------+---------+-------+--------+---------+--------+---------------+---------------+-------+--------+----------------+
|13300000|7420|4       

In [5]:
# 2.1) SALVAR no GCS
blob_csv = bucket.blob(GCS_CSV_OBJECT)
blob_csv.upload_from_filename(str(LOCAL_CSV_PATH))

for file_path in LOCAL_SPARK_OUTPUT.rglob("*"):
    if file_path.is_file():
        rel = file_path.relative_to(LOCAL_SPARK_OUTPUT).as_posix()
        gcs_obj = f"{GCS_PARQUET_PREFIX}/{rel}"
        bucket.blob(gcs_obj).upload_from_filename(str(file_path))

print(f"GCS save OK: gs://{BUCKET_NAME}/{GCS_PREFIX}")

GCS save OK: gs://my_bucket_class_by_vs_code/save-and-catch/20260317_222508


## Explicação detalhada (linha por linha) — Carga para BigQuery

Esta célula faz a etapa **GCS ➜ BigQuery** com fallback seguro.

1. `parquet_uris = [...]`  
   - Lista todos os objetos do bucket no prefixo `GCS_PARQUET_PREFIX`.  
   - Filtra só arquivos que terminam com `.parquet`.  
   - Converte cada objeto em URI completa `gs://...`.

2. `if not parquet_uris:`  
   - Se não encontrou parquet no GCS, entra no plano B.

3. `local_parquet_files = ...`  
   - Verifica se já existe parquet local em `LOCAL_SPARK_OUTPUT`.

4. `if not local_parquet_files:`  
   - Se também não existir localmente, tenta recriar parquet a partir de DataFrame Spark em memória.

5. `if "spark_df_reopened" in globals(): ...`  
   - Prioriza `spark_df_reopened` (quando já houve reabertura do parquet).

6. `elif "spark_df" in globals(): ...`  
   - Usa `spark_df` como alternativa.

7. `else: raise FileNotFoundError(...)`  
   - Falha com mensagem guiando quais células executar antes.

8. `for file_path in LOCAL_SPARK_OUTPUT.rglob("*.parquet"):`  
   - Percorre todos os parquets locais.

9. `rel = file_path.relative_to(...).as_posix()`  
   - Gera caminho relativo (mantendo estrutura de pastas de partição).

10. `gcs_obj = f"{GCS_PARQUET_PREFIX}/{rel}"`  
    - Monta o caminho do objeto no GCS dentro do prefixo da execução.

11. `bucket.blob(gcs_obj).upload_from_filename(...)`  
    - Faz upload de cada parquet para o bucket.

12. `parquet_uris = [...]` (segunda vez)  
    - Reconsulta o GCS para confirmar os arquivos recém-enviados.

13. `if not parquet_uris: raise FileNotFoundError(...)`  
    - Segunda validação: sem parquet no GCS, não segue para BigQuery.

14. `job_config = bigquery.LoadJobConfig(...)`  
    - `source_format=PARQUET`: origem é parquet.  
    - `write_disposition=WRITE_TRUNCATE`: sobrescreve tabela destino.  
    - `autodetect=True`: BigQuery infere schema automaticamente.

15. `load_job = bq_client.load_table_from_uri(...)`  
    - Dispara job de carga no BigQuery usando a lista de URIs.

16. `load_job.result()`  
    - Espera o término do job (bloqueante) e captura erro aqui, se houver.

17. `preview = bq_client.query(... LIMIT 10).to_dataframe()`  
    - Validação rápida consultando 10 linhas da tabela carregada.

18. `print(...)` e `display(preview.head())`  
    - Mostra tabela destino, quantidade de parquets usados e preview dos dados.

In [6]:
# 2.2) SALVAR no BigQuery (tabela final) sem pandas
# Estratégia robusta:
# 1) tenta parquet já no GCS
# 2) se não achar, recria/usa parquet local e faz upload para GCS
# 3) carrega no BigQuery

parquet_uris = [
    f"gs://{BUCKET_NAME}/{blob.name}"
    for blob in storage_client.list_blobs(BUCKET_NAME, prefix=GCS_PARQUET_PREFIX)
    if blob.name.endswith(".parquet")
]

if not parquet_uris:
    print("Nenhum parquet no GCS para este RUN_ID. Tentando enviar do local...")

    # Se não houver arquivos locais, recria a partir do dataframe Spark em memória
    local_parquet_files = list(LOCAL_SPARK_OUTPUT.rglob("*.parquet")) if LOCAL_SPARK_OUTPUT.exists() else []
    if not local_parquet_files:
        if "spark_df_reopened" in globals():
            spark_df_reopened.write.mode("overwrite").parquet(str(LOCAL_SPARK_OUTPUT))
        elif "spark_df" in globals():
            spark_df.write.mode("overwrite").parquet(str(LOCAL_SPARK_OUTPUT))
        else:
            raise FileNotFoundError(
                "Sem parquet no GCS e sem dataframe Spark em memória. "
                "Execute as células 6 e 7 e tente novamente."
            )

    # Upload dos parquet locais para o prefixo atual no GCS
    for file_path in LOCAL_SPARK_OUTPUT.rglob("*.parquet"):
        rel = file_path.relative_to(LOCAL_SPARK_OUTPUT).as_posix()
        gcs_obj = f"{GCS_PARQUET_PREFIX}/{rel}"
        bucket.blob(gcs_obj).upload_from_filename(str(file_path))

    parquet_uris = [
        f"gs://{BUCKET_NAME}/{blob.name}"
        for blob in storage_client.list_blobs(BUCKET_NAME, prefix=GCS_PARQUET_PREFIX)
        if blob.name.endswith(".parquet")
    ]

if not parquet_uris:
    raise FileNotFoundError(
        "Ainda não foram encontrados arquivos parquet no GCS. "
        "Execute as células 6 e 7 e rode esta célula novamente."
    )

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True,
)

load_job = bq_client.load_table_from_uri(
    source_uris=parquet_uris,
    destination=FULL_TABLE_ID,
    job_config=job_config,
)
load_job.result()

preview = bq_client.query(f"SELECT * FROM `{FULL_TABLE_ID}` LIMIT 10").to_dataframe()
print("BigQuery save OK (via Spark parquet no GCS) na tabela:", FULL_TABLE_ID)
print(f"Arquivos parquet carregados: {len(parquet_uris)}")
display(preview.head())

c:\Users\marco\Documents\code_classes\GCP-Data-Engineering-Foundations\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


BigQuery save OK (via Spark parquet no GCS) na tabela: global-grammar-432121-d7.cars_data.housing_data_save_and_catch_test
Arquivos parquet carregados: 1


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


## Explicação detalhada (linha por linha) — Limpeza de artefatos GCP/local

Esta célula remove os artefatos criados na execução para deixar o ambiente limpo.

1. `CONFIRM_DELETE = True`  
   - Chave de segurança para evitar exclusão acidental.

2. `if CONFIRM_DELETE:`  
   - Só executa exclusão quando a confirmação está habilitada.

3. `deleted_gcs = 0`  
   - Inicializa contador de objetos excluídos no GCS.

4. `for b in storage_client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX):`  
   - Lista todos os objetos do bucket dentro do prefixo desta execução.

5. `b.delete()` e `deleted_gcs += 1`  
   - Remove objeto por objeto e atualiza o contador.

6. `bq_client.delete_table(FULL_TABLE_ID, not_found_ok=True)`  
   - Remove a tabela final no BigQuery.  
   - `not_found_ok=True` evita erro se a tabela já não existir.

7. `if LOCAL_SPARK_OUTPUT.exists():`  
   - Só tenta remover parquet local se a pasta existir.

8. `for p in sorted(..., reverse=True): if p.is_file(): p.unlink()`  
   - Remove arquivos primeiro (ordem reversa ajuda em estruturas aninhadas).

9. `for p in sorted(..., reverse=True): if p.is_dir(): p.rmdir()`  
   - Remove diretórios vazios depois de apagar arquivos.

10. `if LOCAL_SPARK_OUTPUT.exists(): LOCAL_SPARK_OUTPUT.rmdir()`  
    - Remove diretório raiz caso ainda exista.

11. `print(...)`  
    - Exibe resumo da limpeza: objetos GCS e tabela BQ removida.

12. `else: print("Exclusão não executada...")`  
    - Mensagem de proteção quando `CONFIRM_DELETE=False`.

In [ ]:
# 3) EXCLUIR artefatos criados (GCS prefix, tabela BQ final e parquet local)
CONFIRM_DELETE = True  # mantenha True para executar exclusão

if CONFIRM_DELETE:
    # GCS: apagar tudo no prefixo desta execução
    deleted_gcs = 0
    for b in storage_client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX):
        b.delete()
        deleted_gcs += 1

    # BigQuery: apagar tabela final
    bq_client.delete_table(FULL_TABLE_ID, not_found_ok=True)

    # Local: apagar pasta parquet
    if LOCAL_SPARK_OUTPUT.exists():
        for p in sorted(LOCAL_SPARK_OUTPUT.rglob("*"), reverse=True):
            if p.is_file():
                p.unlink()
        for p in sorted(LOCAL_SPARK_OUTPUT.rglob("*"), reverse=True):
            if p.is_dir():
                p.rmdir()
        if LOCAL_SPARK_OUTPUT.exists():
            LOCAL_SPARK_OUTPUT.rmdir()

    print("Exclusão concluída.")
    print("Objetos GCS excluídos:", deleted_gcs)
    print("Tabela BQ excluída:", FULL_TABLE_ID)
else:
    print("Exclusão não executada (CONFIRM_DELETE=False).")